# Magic

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import shutil
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OrdinalEncoder, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

import torch
from torch import optim
from torch import nn

from torch.utils.tensorboard import SummaryWriter

from torchmetrics.regression import MeanSquaredError, MeanAbsoluteError
from torchmetrics.classification import Accuracy, AUROC, F1Score, Recall, Precision

from catboost import CatBoostClassifier, CatBoostRegressor, Pool

from modules import LinearRegression, LogisticRegression, TabularModel, TabularDataset

from tqdm.notebook import tqdm

In [3]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [8]:
DATA_PATH = Path('../data')
DATASET_NAME = "magic"

MAGIC_PATH = DATA_PATH / DATASET_NAME / 'magic04.data'

STRATEGY_NUM = "mean"
STRATEGY_CAT = "most_frequent"

RANDOM_STATE = 0
TEST_SIZE = 0.2

In [9]:
def seed_everything(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(RANDOM_STATE)

In [10]:
shutil.rmtree(Path(DATASET_NAME), ignore_errors=True)

In [15]:
magic_names = [
    "fLength",
    "fWidth",
    "fSize",
    "fConc",
    "fConc1",
    "fAsym",
    "fM3Long",
    "fM3Trans",
    "fAlpha",
    "fDist",
    "class"
]

magic_df = pd.read_csv(MAGIC_PATH, header=None, names=magic_names)
magic_df.head()

,fLength,fWidth,fSize,fConc,fConc1,fAsym,fM3Long,fM3Trans,fAlpha,fDist,class
0,28.7967,16.0021,2.6449,0.3918,0.1982,27.7004,22.0110,-8.2027,40.0920,81.8828,g
1,31.6036,11.7235,2.5185,0.5303,0.3773,26.2722,23.8238,-9.9574,6.3609,205.2610,g
2,162.0520,136.0310,4.0612,0.0374,0.0187,116.7410,-64.8580,-45.2160,76.9600,256.7880,g
3,23.8172,9.5728,2.3385,0.6147,0.3922,27.2107,-6.4633,-7.1513,10.4490,116.7370,g
4,75.1362,30.9205,3.1611,0.3168,0.1832,-5.5277,28.5525,21.8393,4.6480,356.4620,g


In [17]:
COLUMNS_NAMES = magic_df.columns.to_list()
N_COLUMNS = len(COLUMNS_NAMES)

CAT_FEATURES = np.array([10])
NUM_FEATURES = np.setdiff1d(np.arange(N_COLUMNS), CAT_FEATURES)

print("Numeric features:", *NUM_FEATURES)
print("Categorical features:", *CAT_FEATURES)

Numeric features: 0 1 2 3 4 5 6 7 8 9
Categorical features: 10


In [18]:
def load_magic() -> TabularDataset:
    magic_df = pd.read_csv(MAGIC_PATH, header=None, names=COLUMNS_NAMES)
    magic_array = magic_df.to_numpy(dtype=object)

    magic_dataset = TabularDataset(magic_array, COLUMNS_NAMES, NUM_FEATURES, CAT_FEATURES)

    return magic_dataset
    

In [ ]:
dataset = load_magic()

## Regression

In [27]:
def get_regression_model(dataset: TabularDataset):
    regression_model = TabularModel()

    regression_model.fit_transformers(dataset, TEST_SIZE, RANDOM_STATE)

    for target_column in range(N_COLUMNS):
        if dataset.is_num_feature(target_column):
            model = LinearRegression(N_COLUMNS - 1).to(device)

            metrics = [
                MeanSquaredError(squared=False).to(device),
                MeanAbsoluteError().to(device)
            ]

        else:
            num_classes = len(regression_model.get_categories(target_column))
            model = LogisticRegression(N_COLUMNS - 1, num_classes).to(device)

            metrics = [
                Accuracy(task='multiclass', num_classes=num_classes).to(device),
                AUROC(task='multiclass', num_classes=num_classes).to(device),
                F1Score(task='multiclass', num_classes=num_classes).to(device),
                Recall(task='multiclass', num_classes=num_classes).to(device),
                Precision(task='multiclass', num_classes=num_classes).to(device)
            ]

        regression_model.models[target_column] = model
        regression_model.metrics[target_column] = metrics
    
    return regression_model

In [28]:
regression_model = get_regression_model(dataset)

Fitting column:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

...

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

In [29]:
def train_regression_model(regression_model: TabularModel,
                            dataset: TabularDataset,
                            epochs: int = 10000):
    
    for target_column in tqdm(range(N_COLUMNS), desc="Column", leave=False):
        X_y_train, X_y_valid = dataset.get_train_test_for_column(target_column, TEST_SIZE, RANDOM_STATE)
        X_y_train = regression_model.transform(X_y_train, target_column, with_target_column=True)
        X_y_valid = regression_model.transform(X_y_valid, target_column, with_target_column=True)

        X_train = X_y_train[:, regression_model.get_mask(target_column)]
        y_train = X_y_train[:, target_column]

        X_valid = X_y_valid[:, regression_model.get_mask(target_column)]
        y_valid = X_y_valid[:, target_column]

        X_train = torch.from_numpy(X_train.astype(float)).to(device, torch.float32)
        X_valid = torch.from_numpy(X_valid.astype(float)).to(device, torch.float32)

        if regression_model.is_num_feature(target_column):
            y_train = torch.from_numpy(y_train.astype(float)).to(device, torch.float32)
            y_valid = torch.from_numpy(y_valid.astype(float)).to(device, torch.float32)
            criterion = nn.MSELoss()
        else:
            y_train = torch.from_numpy(y_train.astype(int)).to(torch.long)
            y_valid = torch.from_numpy(y_valid.astype(int)).to(torch.long)
            criterion = nn.CrossEntropyLoss()
        
        # ------------------- TRAINING ----------------------
        model = regression_model.models[target_column]
        optimizer = optim.Adam(model.parameters(), lr=1e-3)

        best_val_loss = float("inf")
        best_state_dict = None
        best_epoch = -1
        
        for epoch in tqdm(range(1, epochs + 1), "Training", leave=False):
            # ---------------------- TRAIN ------------------------
            model.train()

            optimizer.zero_grad()
            prediction = model(X_train)
            loss = criterion(prediction, y_train)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            # ---------------------- VALID -------------------------
            model.eval()
            with torch.no_grad():
                pred_valid = model(X_valid)
                val_loss = criterion(pred_valid, y_valid).item()

                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_epoch = epoch
                    best_state_dict = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if best_state_dict is not None:
            model.load_state_dict(best_state_dict)
        print(f"Target column = {target_column}, "
              f"Type = {model.__class__.__name__ }, "
              f"Best epoch = {best_epoch}, "
              f"Best val loss={best_val_loss:.6f}")

In [30]:
train_regression_model(regression_model, dataset, epochs=10000)

Column:   0%|          | 0/12 [00:00<?, ?it/s]

Training:   0%|          | 0/10000 [00:00<?, ?it/s]

Target column = 0, Type = LinearRegression, Best epoch = 1299, Best val loss=0.991305


Training:   0%|          | 0/10000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [32]:
def bootstrap_regression(regression_model: TabularModel,
                         dataset: TabularDataset,
                         iterations: int = 1000):
    
    output = list()
    
    for target_column in tqdm(range(N_COLUMNS), desc="Column", leave=False):

        model = regression_model.models[target_column]
        metrics = regression_model.metrics[target_column]
        metrics_by_name = defaultdict(list)

        _, X_y_test = dataset.get_train_test_for_column(target_column, test_size=0.2, random_state=RANDOM_STATE)
        X_y_test = regression_model.transform(X_y_test, target_column, with_target_column=True)
        
        for metric in metrics:
            name = metric.__class__.__name__

            for _ in tqdm(range(iterations), desc="Bootstrap", leave=False):
                bootstrap = np.random.choice(X_y_test.shape[0], X_y_test.shape[0], replace=True)
                X_test = X_y_test[:, regression_model.get_mask(target_column)]
                X_test = X_test[bootstrap, :]
                y_test = X_y_test[bootstrap, target_column]
                y_test = y_test[bootstrap]

                X_test = torch.from_numpy(X_test.astype(float)).to(device, torch.float32)
                if regression_model.is_num_feature(target_column):
                    y_pred = model.predict(X_test)
                    y_test = torch.from_numpy(y_test.astype(float)).to(device, torch.float32)
                else:
                    y_pred = model.predict_proba(X_test)
                    y_test = torch.from_numpy(y_test.astype(int)).to(device, torch.long)

                value = metric(y_pred, y_test)
                metrics_by_name[name].append(value.item())
        
        for name, values in metrics_by_name.items():
            values = np.array(values, dtype=float)

            row = {
                "column": dataset.columns_names[target_column],
                "model": "regression",
                "metric": name,
                "mean": values.mean(),
                "std": values.std(ddof=1) if values.size > 1 else 0.0
            }

            output.append(row)

    output = pd.DataFrame(output)
    return output

In [33]:
regression_model_statistics = bootstrap_regression(regression_model, dataset, iterations=1000)

Column:   0%|          | 0/12 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

## CatBoost

In [34]:
def get_catboost(dataset: TabularDataset, iterations: int = 100, learning_rate: float = 1e-3):
    catboost_model = TabularModel()

    catboost_model.fit_transformers(dataset, TEST_SIZE, RANDOM_STATE)

    task_type = "GPU" if torch.cuda.is_available() else "CPU"

    for target_column in range(N_COLUMNS):
        if dataset.is_num_feature(target_column):
            model = CatBoostRegressor(iterations=iterations,
                                      learning_rate=learning_rate,
                                      task_type=task_type,
                                      allow_writing_files=False)
            
            metrics = [
                MeanSquaredError(squared=False).to(device),
                MeanAbsoluteError().to(device)
            ]

        else:
            model = CatBoostClassifier(iterations=iterations,
                                       learning_rate=learning_rate,
                                       task_type=task_type,
                                       allow_writing_files=False)
            
            num_classes = len(regression_model.get_categories(target_column))
            metrics = [
                Accuracy(task='multiclass', num_classes=num_classes).to(device),
                AUROC(task='multiclass', num_classes=num_classes).to(device),
                F1Score(task='multiclass', num_classes=num_classes).to(device),
                Recall(task='multiclass', num_classes=num_classes).to(device),
                Precision(task='multiclass', num_classes=num_classes).to(device)
            ]

        catboost_model.models[target_column] = model
        catboost_model.metrics[target_column] = metrics
    
    return catboost_model

In [35]:
catboost_model = get_catboost(dataset, iterations=100)

Fitting column:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

...

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

Fitting column within:   0%|          | 0/12 [00:00<?, ?it/s]

In [36]:
def train_catboost_model(catboost_model: TabularModel,
                         dataset: TabularDataset):
    
    for target_column in tqdm(range(N_COLUMNS), desc="Column", leave=False):
        X_y_train, X_y_valid = dataset.get_train_test_for_column(target_column, TEST_SIZE, RANDOM_STATE)

        X_y_train = catboost_model.transform(X_y_train, target_column, impute_num_features=False)
        X_y_valid = catboost_model.transform(X_y_valid, target_column, impute_num_features=False)

        X_train = X_y_train[:, regression_model.get_mask(target_column)]
        y_train = X_y_train[:, target_column]
        X_valid = X_y_valid[:, regression_model.get_mask(target_column)]
        y_valid = X_y_valid[:, target_column]

        cat_features = regression_model.get_shifted_cat_features(target_column)

        train_pool = Pool(X_train, y_train, cat_features)
        valid_pool = Pool(X_valid, y_valid, cat_features)
        
        catboost_model.models[target_column].fit(train_pool, eval_set=valid_pool, use_best_model=True, verbose=False)

In [37]:
train_catboost_model(catboost_model, dataset)

Column:   0%|          | 0/12 [00:00<?, ?it/s]

In [38]:
def bootstrap_catboost(catboost_model: TabularModel,
                       dataset: TabularDataset,
                       iterations: int = 1000):
    
    output = list()
    
    for target_column in tqdm(range(N_COLUMNS), desc="Column", leave=False):

        model = catboost_model.models[target_column]
        metrics = catboost_model.metrics[target_column]
        metrics_by_name = defaultdict(list)

        _, X_y_test = dataset.get_train_test_for_column(target_column, TEST_SIZE, RANDOM_STATE)
        X_y_test = catboost_model.transform(X_y_test, target_column, impute_num_features=False)

        for metric in metrics:
            name = metric.__class__.__name__

            for _ in tqdm(range(iterations), desc="Bootstrap", leave=False):
                bootstrap = np.random.choice(X_y_test.shape[0], X_y_test.shape[0], replace=True)
                X_test = X_y_test[:, catboost_model.get_mask(target_column)]
                X_test = X_test[bootstrap, :]
                y_test = X_y_test[bootstrap, target_column]
                y_test = y_test[bootstrap]

                if catboost_model.is_num_feature(target_column):
                    y_pred = model.predict(X_test)
                    y_pred = torch.from_numpy(y_pred.astype(float)).to(device, dtype=torch.float32)
                    y_test = torch.from_numpy(y_test.astype(float)).to(device, dtype=torch.float32)
                else:
                    y_pred = model.predict_proba(X_test)
                    y_pred = torch.from_numpy(y_pred.astype(float)).to(device, dtype=torch.float32)
                    y_test = torch.from_numpy(y_test.astype(int)).to(device, dtype=torch.long)

                value = metric(y_pred, y_test)
                metrics_by_name[name].append(value.item())

        for name, values in metrics_by_name.items():
            values = np.array(values, dtype=float)

            row = {
                "column": dataset.columns_names[target_column],
                "model": "catboost",
                "metric": name,
                "mean": values.mean(),
                "std": values.std(ddof=1) if values.size > 1 else 0.0
            }

            output.append(row)
    
    output = pd.DataFrame(output)
    return output

In [39]:
catboost_model_statistics = bootstrap_catboost(catboost_model, dataset, iterations=1000)

Column:   0%|          | 0/12 [00:00<?, ?it/s]

Bootstrap:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

# Compare

In [28]:
def get_compare_table(regression_model_statistics: pd.DataFrame, catboost_model_statistics: pd.DataFrame):
    out = pd.concat([regression_model_statistics, catboost_model_statistics], axis=0)
    out = out.set_index(["column", "metric", "model"]).sort_index()
    out = out.round(5)
    return out

In [ ]:
catboost_vs_regression = get_compare_table(regression_model_statistics, catboost_model_statistics)
catboost_vs_regression.to_excel(f"{DATASET_NAME}_regression_vs_catboost.xlsx", index=True)

In [30]:
catboost_vs_regression.head(15)

mean      std
column       metric             model                       
age          MeanAbsoluteError  catboost    0.82731  0.00978
                                regression  0.88032  0.01135
             MeanSquaredError   catboost    1.00569  0.01181
                                regression  1.07971  0.01301
capital-gain MeanAbsoluteError  catboost    0.26523  0.01680
                                regression  0.30205  0.01719
             MeanSquaredError   catboost    0.96948  0.11039
                                regression  1.00233  0.10682
capital-loss MeanAbsoluteError  catboost    0.40920  0.01573
                                regression  0.40921  0.01612
             MeanSquaredError   catboost    0.97590  0.04124
                                regression  0.99268  0.03869
class        MulticlassAUROC    catboost    0.50043  0.01222
                                regression  0.49956  0.01172
             MulticlassAccuracy catboost    0.67495  0.00827